## Below cell is mandatory to get the catalog and schema access

In [0]:
%sql
USE CATALOG dbacademy;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS dbacademy.healthcare;
USE SCHEMA healthcare;

In [0]:
%sql
SELECT current_catalog(),current_schema();

create a volume healthcard_raw

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS  dbacademy.healthcare.healthcare_raw;

**Create a volume health_care under labuser123232.. schema, Upload patients.csv, prescriptions.csv and encounters.csv to your  /Volumes/DA.catalog_name/DA.schema_name/healthcare_raw**

In [0]:
dbutils.fs.ls('/Volumes/dbacademy/healthcare/healthcare_raw')

Write a COPY INTO to load it into delta table name patients, encounters and prescriptions from above csv files

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dbacademy.healthcare.patients;

COPY INTO dbacademy.healthcare.patients
FROM '/Volumes/dbacademy/healthcare/healthcare_raw/patients.csv'
FILEFORMAT = CSV
COPY_OPTIONS ('skipRows' = '1',      -- Skips the first row
  'header'   = 'true',   -- Use second row as header
  'delimiter' = ','   ,   -- Set delimiter if needed
  'mergeSchema' = 'true'
  )


In [0]:
%sql
DROP TABLE IF EXISTS dbacademy.healthcare.encounters;
CREATE TABLE IF NOT EXISTS dbacademy.healthcare.encounters;

COPY INTO dbacademy.healthcare.encounters
FROM '/Volumes/dbacademy/healthcare/healthcare_raw/encounters.csv'
FILEFORMAT = CSV
COPY_OPTIONS ('skipRows' = '1',      -- Skips the first row
  'header'   = 'true',   -- Use second row as header
  'delimiter' = ','   ,   -- Set delimiter if needed
  'mergeSchema' = 'true'
  )


In [0]:
df=spark.sql("""SELECT * FROM dbacademy.healthcare.encounters""")
df= df.drop('_c0',
 '_c1',
 '_c2',
 '_c3',
 '_c4',
 '_c5',
 '_c6',
 '_c7',
 '_c8',
 '_c9',
 '_c10',
 '_c11',
 '_c12',
 '_c13')

In [0]:
%sql
select * from dbacademy.healthcare.encounters;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dbacademy.healthcare.prescriptions;

COPY INTO dbacademy.healthcare.prescriptions
FROM '/Volumes/dbacademy/healthcare/healthcare_raw/prescriptions.csv'
FILEFORMAT = CSV
COPY_OPTIONS ('mergeSchema' = 'true')


**verify the number of rows in each tables**

In [0]:
patients= spark.sql("SELECT count(*) as total_rows FROM dbacademy.healthcare.patients")
encounters= spark.sql("SELECT count(*) as total_rows FROM dbacademy.healthcare.encounters")
prescriptions= spark.sql("SELECT count(*) as total_rows FROM dbacademy.healthcare.prescriptions")
df = patients.union(encounters).union(prescriptions)
display(df)

Load prescription_batch2_copy.csv(50 rows) into DA.catalog_name.DA.schema_name.prescriptions

In [0]:
spark.sql("""
COPY INTO dbacademy.healthcare.prescriptions
FROM '/Volumes/dbacademy/healthcare/healthcare_raw/prescriptions_batch2_copy_into.csv'
FILEFORMAT = CSV
COPY_OPTIONS ('mergeSchema' = 'true')
""")

#### Verify how many rows are there in the prescriptions table after loading

In [0]:
df = spark.sql("SELECT count(*) as total_rows FROM dbacademy.healthcare.prescriptions")
display(df)

add new column name triage_score of type int to the encounters table 

In [0]:
spark.sql("""
ALTER TABLE dbacademy.healthcare.encounters
ADD COLUMN triage_score INT
""")

upload the encounters_vs_schema_evoluton.csv and add these data into the encounters table. 

In [0]:
spark.sql("""
COPY INTO dbacademy.healthcare.encounters
FROM '/Volumes/dbacademy/healthcare/healthcare_raw/encounters_v2_schema_evolution.csv'
FILEFORMAT = CSV
COPY_OPTIONS ('mergeSchema' = 'true')
""")

Run describe table extended and list the new columns

In [0]:
df = spark.sql("DESCRIBE TABLE EXTENDED dbacademy.healthcare.encounters")
display(df[df['col_name'].isin(['triage_score'])])

Extract sensor_id from rescured column

In [0]:

df = df.drop('_c0',
 '_c1',
 '_c2',
 '_c3',
 '_c4',
 '_c5',
 '_c6',
 '_c7',
 '_c8',
 '_c9',
 '_c10',
 '_c11',
 '_c12',
 '_c13')
display(df)

patients_merge_source.csv contains 30 updated records and 5 new patients. Write acode to perfrom upsert this cell to patients table.

In [0]:
spark.sql("""
MERGE INTO dbacademy.healthcare.patients AS target
USING (
  SELECT * FROM '/Volumes/dbacademy/healthcare/healthcare_raw/patients_merge_source.csv'
) AS source
ON target.patient_id = source.patient_id
WHEN MATCHED THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *
""")

In [0]:
%sql
--write sql code to get this report shown below

load patient_vitals_variant.csv into delta table. Cast the vitals_json column to variat type and create the table with that schema patient_id String, recorded_at Date, vitals variant.

extreact bp_systolic, heart_rate, and spo2_pct from the variant columnand flg patients whtere spo2 is below 94

In [0]:
#python/sql code to implement the below logic
